In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

tf.keras.utils.set_random_seed(42)

data = fetch_openml(name="sonar", version=1, as_frame=False)
X = data.data
y = LabelEncoder().fit_transform(data.target)

print("Dataset shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model1 = keras.Sequential([
    layers.Input(shape=(60,)),
    layers.Dense(512, activation='relu', use_bias=True),
    layers.Dense(256, activation='relu', use_bias=True),
    layers.Dense(128, activation='relu', use_bias=True),
    layers.Dense(64, activation='relu', use_bias=True),
    layers.Dense(32, activation='relu', use_bias=True),
    layers.Dense(1, activation='sigmoid', use_bias=True)
])

model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model2 = keras.Sequential([
    layers.Input(shape=(60,)),
    layers.Dense(512, activation='relu', use_bias=True),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu', use_bias=True),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu', use_bias=True),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu', use_bias=True),
    layers.Dense(32, activation='relu', use_bias=True),
    layers.Dense(1, activation='sigmoid', use_bias=True)
])

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model3 = keras.Sequential([
    layers.Input(shape=(60,)),
    layers.Dense(512, activation='relu', use_bias=True,
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(256, activation='relu', use_bias=True,
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(128, activation='relu', use_bias=True,
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(64, activation='relu', use_bias=True,
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(32, activation='relu', use_bias=True,
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1, activation='sigmoid', use_bias=True)
])

model3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history1 = model1.fit(
    X_train, y_train, epochs=250, batch_size=8,
    validation_split=0.25, verbose=0
)

history2 = model2.fit(
    X_train, y_train, epochs=250, batch_size=8,
    validation_split=0.25, verbose=0
)

history3 = model3.fit(
    X_train, y_train, epochs=250, batch_size=8,
    validation_split=0.25, verbose=0
)

_, acc1 = model1.evaluate(X_test, y_test, verbose=0)
_, acc2 = model2.evaluate(X_test, y_test, verbose=0)
_, acc3 = model3.evaluate(X_test, y_test, verbose=0)

print("\nTest Accuracy")
print("--------------------------")
print("Without Regularization:", round(acc1, 3))
print("Dropout               :", round(acc2, 3))
print("L2 Regularization     :", round(acc3, 3))

plt.figure(figsize=(9, 5))
plt.plot(history1.history['val_accuracy'],
         label='Without Regularization')
plt.plot(history2.history['val_accuracy'],
         label='Dropout')
plt.plot(history3.history['val_accuracy'],
         label='L2 Regularization')

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Regularization on Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
